In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Pick a starting note and drag $k$. The gold path traces the $k$ steps along
the curve, from the blue start to the red target. The dotted lines drop to
the frequency axis, so you can see the jump in Hertz grow as you move up.
The audio card plays the start, then the target, then both.

In [ ]:
# hide
# autorun
NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]

def name(p, NAMES=NAMES):
    return f"{NAMES[p % 12]}{p // 12 - 1}"

def freq(p):
    return 440.0 * 2 ** ((p - 69) / 12)

P_LO, P_HI = 33, 81                  # A1 (55 Hz) to A5 (880 Hz)
P_ALL = np.arange(P_LO, P_HI + 1)
P_CURVE = np.linspace(P_LO, P_HI, 400)
START0, K0 = 57, 7                   # starting parameters: A3, up seven steps
Y_FLOOR = 30

SR = 44100
def tone(f, dur, sr=SR):
    # a few harmonics, so the low notes still come through small speakers
    t = np.arange(int(dur * sr)) / sr
    x = sum(np.sin(2 * np.pi * h * f * t) / h for h in range(1, 5))
    ramp = int(0.01 * sr)
    x[:ramp] *= np.linspace(0, 1, ramp)
    x[-ramp:] *= np.linspace(1, 0, ramp)
    return x

def figure():
    fig = go.Figure()
    p0, p1 = START0, START0 + K0
    path = np.linspace(p0, p1, 60)
    fig.add_scatter(x=freq(P_CURVE), y=P_CURVE, mode="lines",
                    line=dict(color=STEEL, width=1.6))
    fig.add_scatter(x=freq(P_ALL), y=P_ALL, mode="markers",
                    marker=dict(color=STEEL, size=6))
    fig.add_scatter(x=freq(path), y=path, mode="lines",
                    line=dict(color=GOLD, width=4))
    fig.add_scatter(x=[freq(p0), freq(p0), None, freq(p1), freq(p1), None],
                    y=[Y_FLOOR, p0, None, Y_FLOOR, p1, None], mode="lines",
                    line=dict(color=IRON, width=1.2, dash="dot"))
    fig.add_scatter(x=[freq(p0)], y=[p0], mode="markers",
                    marker=dict(color=BLUE, size=13, line=dict(color="white", width=2)))
    fig.add_scatter(x=[freq(p1)], y=[p1], mode="markers",
                    marker=dict(color=RED, size=13, line=dict(color="white", width=2)))
    fig.update_xaxes(range=[0, 920], title_text="Frequency (Hz)", fixedrange=True)
    fig.update_yaxes(range=[Y_FLOOR, 84], title_text="MIDI pitch", fixedrange=True)
    return fig

def controls(fig):
    start = widgets.SelectionSlider(description="Start note",
                                    options=[(name(p), p) for p in range(45, 70)],
                                    value=START0)
    k = widgets.IntSlider(description="Steps $k$", min=-12, max=12, value=K0)
    readout = widgets.HTML()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(p0, k, freq=freq, name=name, Y_FLOOR=Y_FLOOR, readout=readout):
        p1 = p0 + k
        f0, f1 = freq(p0), freq(p1)
        path = np.linspace(p0, p1, 60)
        with fig.batch_update():
            fig.data[2].x, fig.data[2].y = freq(path), path
            fig.data[3].x = [f0, f0, None, f1, f1, None]
            fig.data[3].y = [Y_FLOOR, p0, None, Y_FLOOR, p1, None]
            fig.data[4].x, fig.data[4].y = [f0], [p0]
            fig.data[5].x, fig.data[5].y = [f1], [p1]
        readout.value = (f"<span style='font-size:0.9em'>{name(p0)} "
                         f"({f0:.1f} Hz) &times; 2<sup>{k}/12</sup> = "
                         f"{name(p1)} ({f1:.1f} Hz) &nbsp;·&nbsp; "
                         f"a jump of {f1 - f0:+.1f} Hz</span>")

    widgets.interactive_output(update, {"p0": start, "k": k})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(freq=freq, tone=tone, SR=SR):
        f0, f1 = freq(start.value), freq(start.value + k.value)
        x = np.concatenate([tone(f0, 0.6), tone(f1, 0.6),
                            tone(f0, 1.0) + tone(f1, 1.0)])
        x *= 0.125 / np.abs(x).max()          # about -18 dBFS, a safe level
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)


    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (start, k):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([start, k, readout, out, gate])

icm_plotly.show(figure, controls)